# Modelling

In [26]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import r2_score, mean_absolute_error
df = pd.read_pickle("../data/processed/df.pkl")

## Train-test split

In [27]:
X = df[[col for col in df.columns if col not in ['ground_truth_people', 'occupancy_ratio', 'timestamp']]]
y_both = df[['ground_truth_people', 'occupancy_ratio']]
X_train, X_test, y_train, y_test = train_test_split(X, y_both, test_size=0.2, random_state=42)
y_train_people = y_train['ground_truth_people'].squeeze()
y_test_people = y_test['ground_truth_people'].squeeze()
y_train_ratio = y_train['occupancy_ratio'].squeeze()
y_test_ratio = y_test['occupancy_ratio'].squeeze()

## Feature selection

In [28]:
rf_ratio = RandomForestRegressor(n_estimators=100, random_state=42)
rf_ratio.fit(X_train, y_train_ratio)
feature_importances_ratio = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf_ratio.feature_importances_
}).sort_values('importance', ascending=False)

rf_people = RandomForestRegressor(n_estimators=100, random_state=42)
rf_people.fit(X_train, y_train_people)
feature_importances_people = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf_people.feature_importances_
}).sort_values('importance', ascending=False)

print("--- Ratio ---")
print(feature_importances_ratio)
print("\n--- People ---")
print(feature_importances_people)


--- Ratio ---
                 feature  importance
12          max_capacity    0.357983
4         probes_rssi_70    0.119448
5         probes_rssi_75    0.112541
6         probes_rssi_80    0.106426
1         probes_rssi_55    0.071076
3         probes_rssi_65    0.043500
2         probes_rssi_60    0.040262
10      unique_mac_4plus    0.032854
9       unique_mac_3plus    0.027654
0           total_probes    0.022745
8       unique_mac_2plus    0.022005
11      unique_mac_5plus    0.021849
7   unique_mac_addresses    0.021657

--- People ---
                 feature  importance
12          max_capacity    0.231477
8       unique_mac_2plus    0.150384
7   unique_mac_addresses    0.149467
0           total_probes    0.140297
9       unique_mac_3plus    0.097901
6         probes_rssi_80    0.078906
5         probes_rssi_75    0.049883
4         probes_rssi_70    0.039090
10      unique_mac_4plus    0.023281
3         probes_rssi_65    0.014705
2         probes_rssi_60    0.010184
1       

## Train + cross validate models

In [29]:
def drop_unimportant_features(X, feature_importances, threshold=0.05):
    important_features = feature_importances[
        feature_importances['importance'] > threshold
    ]['feature'].tolist()
    return X[important_features]

def evaluate_model(model, X_train, y_train, model_name, target_name):
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
    mae_scores = cross_val_score(model, X_train, y_train, cv=5,
                                  scoring='neg_mean_absolute_error')
    return scores.mean(), (-mae_scores).mean(), scores.std(), (-mae_scores).std()

models = {
    'Ridge': Ridge(),
    'Decision Tree': DecisionTreeRegressor(max_depth=4, min_samples_leaf=5, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=5, 
                                            min_samples_leaf=5, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=3, 
                                                    learning_rate=0.1, random_state=42)
}


## Cross validation score comparison 

In [30]:
X_people = drop_unimportant_features(X_train, feature_importances_people, threshold=0.05)
X_ratio = drop_unimportant_features(X_train, feature_importances_ratio, threshold=0.05)
results = []
for name, model in models.items():
    r2_p, mae_p, std_p, std_mae_p = evaluate_model(model, X_people, y_train_people, name, 'people')
    r2_r, mae_r, std_r, std_mae_r = evaluate_model(model, X_ratio, y_train_ratio, name, 'ratio')
    results.append({
        'model': name,
        'people_r2': r2_p, 'people_r2_std': std_p, 'people_mae': mae_p, 'people_mae_std': std_mae_p,
        'ratio_r2': r2_r, 'ratio_r2_std': std_r, 'ratio_mae': mae_r, 'ratio_mae_std': std_mae_r
    })

results_df = pd.DataFrame(results)
results_df

,model,people_r2,people_r2_std,people_mae,people_mae_std,ratio_r2,ratio_r2_std,ratio_mae,ratio_mae_std
0,Ridge,0.855087,0.028584,13.196627,3.037810,0.706180,0.073446,0.066641,0.015260
1,Decision Tree,0.841190,0.072332,12.112743,4.907111,0.773422,0.078317,0.058012,0.004851
2,Random Forest,0.815719,0.058019,11.831691,5.456657,0.775101,0.077171,0.059696,0.005304
3,Gradient Boosting,0.952978,0.031814,6.506032,2.495249,0.810602,0.067642,0.053117,0.009999


## Hyperparameter tuning for chosen model

In [31]:
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [2, 3],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8]
}

scoring = {
    'mae': 'neg_mean_absolute_error',
    'r2': 'r2'
}

# 1. Tuning for predicting 'ground_truth_people'
grid_search_people = GridSearchCV(
    GradientBoostingRegressor(random_state=42), 
    param_grid, 
    cv=5, 
    scoring=scoring,
    refit='mae',
    n_jobs=-1
)
grid_search_people.fit(X_people, y_train_people)
gb_best_people = grid_search_people.best_estimator_

idx_p = grid_search_people.best_index_
best_mae_p = -grid_search_people.cv_results_['mean_test_mae'][idx_p]
best_r2_p = grid_search_people.cv_results_['mean_test_r2'][idx_p]

# 2. Tuning for predicting 'occupancy_ratio'
grid_search_ratio = GridSearchCV(
    GradientBoostingRegressor(random_state=42), 
    param_grid, 
    cv=5, 
    scoring=scoring,
    refit='mae',
    n_jobs=-1
)
grid_search_ratio.fit(X_ratio, y_train_ratio)
gb_best_ratio = grid_search_ratio.best_estimator_

idx_r = grid_search_ratio.best_index_
best_mae_r = -grid_search_ratio.cv_results_['mean_test_mae'][idx_r]
best_r2_r = grid_search_ratio.cv_results_['mean_test_r2'][idx_r]

print(f"People - Best params: {grid_search_people.best_params_}")
print(f"         CV MAE: {best_mae_p:.3f} people | CV R²: {best_r2_p:.3f}\n")

print(f"Ratio  - Best params: {grid_search_ratio.best_params_}")
print(f"         CV MAE: {best_mae_r:.3f} ratio  | CV R²: {best_r2_r:.3f}")

People - Best params: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.8}
         CV MAE: 6.184 people | CV R²: 0.961

Ratio  - Best params: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 50, 'subsample': 0.7}
         CV MAE: 0.052 ratio  | CV R²: 0.818


## Evaluation on testing set

In [32]:
gb_best_people = grid_search_people.best_estimator_
gb_best_ratio = grid_search_ratio.best_estimator_

X_test_people = drop_unimportant_features(X_test, feature_importances_people, threshold=0.05)
X_test_ratio = drop_unimportant_features(X_test, feature_importances_ratio, threshold=0.05)

y_pred_people = gb_best_people.predict(X_test_people)
y_pred_ratio = gb_best_ratio.predict(X_test_ratio)

test_results = pd.DataFrame([
    {
        'Target': 'People (ground_truth_people)',
        'Test R²': r2_score(y_test_people, y_pred_people),
        'Test MAE': mean_absolute_error(y_test_people, y_pred_people)
    },
    {
        'Target': 'Ratio (occupancy_ratio)',
        'Test R²': r2_score(y_test_ratio, y_pred_ratio),
        'Test MAE': mean_absolute_error(y_test_ratio, y_pred_ratio)
    }
])

test_results

,Target,Test R²,Test MAE
0,People (ground_truth_people),0.895605,7.040660
1,Ratio (occupancy_ratio),0.829246,0.060671


In [33]:
model_filename = "../models/gb_best_people.joblib"
joblib.dump(gb_best_people, model_filename)
print(f"Model successfully saved to: {model_filename}")

feature_list = X_people.columns.tolist()
features_filename = "../models/selected_features_people.joblib"
joblib.dump(feature_list, features_filename)
print(f"Feature columns saved to: {features_filename}")

Model successfully saved to: ../models/gb_best_people.joblib
Feature columns saved to: ../models/selected_features_people.joblib


## Summary

### Target variable selection: Headcount ``ground_truth_people``
Two potential modeling paths were evaluated based on target variable definitions: tracking the literal headcount (People) versus tracking a relative percentage (Ratio).

We have explicitly selected People ```ground_truth_people``` as the production target variable based on two factors:
- Superior unseen performance: r² = 0.896 > r² = 0.829
- Operational Actionability: 
    - Predicting raw headcounts yields an immediate, physical metricindicating 
    - Predictions deviate by roughly 7 people on average

### Algorithm selection: Gradient Boosted Regressor
Outperformed all other algorithms (Ridge, Decision Tree, Random Forest) for all metrics in both target variables.


### Overfitting mitigation strategy
- Controlled Tree Growth: Restricted the maximum depth to 3 (max_depth=3) to prevent individual trees from building
over-complicated rules.
- Introduced Random Sampling: Set the model to train on a random 80% subset of rows for each tree (subsample=0.8) to break up data memorization.
- Controlled Learning Rate: Used a lower learning rate of 0.05 across 100 trees to ensure stable, incremental pattern learning.
- Achieved Ideal Sweet Spot: Successfully closed the generalization gap to a healthy 6.5%, keeping test performance extremely high at an 0.896 \[R^{2}\].